In [1]:
import logging
import random
import time

import pymysql
from faker import Faker


In [2]:
def get_connection(config: dict) -> pymysql.Connection:
    host = config.get("host", "localhost")
    user = config.get("user", "root")
    password = config.get("password", "")
    database = config.get("database")
    port = config.get("port", 3306)
    charset = config.get("charset", "utf8mb4")
    autocommit = config.get("autocommit", False)
    return pymysql.connect(user=user, password=password, host=host, database=database, port=port, charset=charset, autocommit=autocommit)


def create_users(cur, fake: Faker, count: int) -> list[int]:
    user_ids: list[int] = []
    for _ in range(count):
        cur.execute("INSERT INTO users (email, name) VALUES (%s, %s)", (fake.unique.email(), fake.name()))
        user_ids.append(cur.lastrowid)
    return user_ids


def create_products_and_skus(cur, fake: Faker, product_count: int, sku_range=(1, 3)) -> list[dict]:
    sku_rows: list[dict] = []
    categories = ["BOOK", "IT", "FOOD", "EDU", "ETC"]
    for _ in range(product_count):
        cur.execute("INSERT INTO products (name, category, status) VALUES (%s, %s, 'ACTIVE')", (fake.catch_phrase(), random.choice(categories)))
        product_id = cur.lastrowid
        sku_count = random.randint(*sku_range)
        for i in range(sku_count):
            price = random.randint(5_000, 50_000)
            stock = random.randint(10, 200)
            cur.execute("INSERT INTO skus (product_id, sku_code, option_text, price, stock, status) VALUES (%s, %s, %s, %s, %s, 'ACTIVE')",
                        (product_id, fake.unique.bothify("SKU-####-????"), f"Option-{i + 1}", price, stock))
            sku_rows.append({"sku_id": cur.lastrowid, "price": price})
    return sku_rows


def create_orders_and_items(cur, fake: Faker, order_count: int, user_ids: list[int], sku_rows: list[dict]) -> list[int]:
    order_ids: list[int] = []
    for _ in range(order_count):
        user_id = random.choice(user_ids)
        order_no = fake.unique.bothify("ORD-########")
        cur.execute("INSERT INTO orders (order_no, user_id, status) VALUES (%s, %s, 'CREATED')", (order_no, user_id))
        order_id = cur.lastrowid
        order_ids.append(order_id)
        total_amount = 0
        item_count = random.randint(1, min(3, len(sku_rows)))
        picked_skus = random.sample(sku_rows, item_count)
        for sku in picked_skus:
            qty = random.randint(1, 3)
            unit_price = sku["price"]
            line_amount = unit_price * qty
            total_amount += line_amount
            cur.execute("INSERT INTO order_items(order_id, sku_id, product_name_snapshot, option_snapshot, unit_price_snapshot, qty, line_amount)VALUES (%s, %s, %s, %s, %s, %s, %s)",
                        (order_id, sku["sku_id"], fake.word().upper(), "Sample Option", unit_price, qty, line_amount))

        cur.execute("UPDATE orders SET total_amount=%s, status='PAID' WHERE order_id=%s", (total_amount, order_id))

    return order_ids


def create_payments(cur, order_ids: list[int]):
    for order_id in order_ids:
        cur.execute("SELECT total_amount FROM orders WHERE order_id=%s", (order_id,))
        amount = cur.fetchone()[0]
        cur.execute("INSERT INTO payments (order_id, method, status, amount, paid_at)VALUES (%s, 'CARD', 'APPROVED', %s, NOW())", (order_id, amount))


def create_shipments(cur, fake: Faker, order_ids: list[int]):
    carriers = ["CJ", "LOTTE", "HANJIN"]
    for order_id in order_ids:
        cur.execute(
            "INSERT INTO shipments (order_id, carrier, tracking_no, status, shipped_at) VALUES (%s, %s, %s, 'IN_TRANSIT', NOW())",
            (order_id, random.choice(carriers), fake.unique.bothify("TRK##########")))


def create_event_logs(cur, fake: Faker, user_ids: list[int], event_count: int = 300):
    event_types = [("USER_SIGNUP", "USER"), ("ORDER_CREATED", "ORDER"), ("PAYMENT_APPROVED", "PAYMENT"), ("SHIPMENT_CREATED", "SHIPMENT")]
    for _ in range(event_count):
        event_type, entity_type = random.choice(event_types)
        cur.execute(
            "INSERT INTO event_log(event_type, entity_type, entity_id, user_id, payload) VALUES (%s, %s, %s, %s, JSON_OBJECT('msg', %s))",
            (event_type, entity_type, random.randint(1, 10_000), random.choice(user_ids), fake.sentence())
        )


def seed_once(fake: Faker, config: dict, user_count: int = 3, product_count: int = 2, sku_range=(1, 3), order_count: int = 5, event_count: int = 50):
    connection = get_connection(config)
    cursor = connection.cursor()
    try:
        user_ids = create_users(cursor, fake, user_count)
        sku_rows = create_products_and_skus(cursor, fake, product_count, sku_range)
        order_ids = create_orders_and_items(cursor, fake, order_count, user_ids, sku_rows)
        create_payments(cursor, order_ids)
        create_shipments(cursor, fake, order_ids)
        create_event_logs(cursor, fake, user_ids, event_count)
        connection.commit()
        logging.info(f"✔ batch inserted: users={len(user_ids)}, products={product_count}, "f"orders={len(order_ids)}, events={event_count}")
    except Exception as e:
        connection.rollback()
        logging.info("✖ batch failed:", repr(e))
        raise
    finally:
        cursor.close()
        connection.close()


def run_loop(fake: Faker, config: dict, repeat: int, sleep: float, user_count: int = 3, product_count: int = 2, sku_range=(1, 3), order_count: int = 5, event_count: int = 50):
    for i in range(1, repeat + 1):
        logging.info(f"\n=== batch {i}/{repeat} ===")
        seed_once(fake, config, user_count, product_count, sku_range, order_count, event_count)
        if i < repeat:
            time.sleep(sleep)

## Mysql-Primary Faker 데이터 입력

In [3]:
run_loop(Faker("ko_KR"), {"host": "mysql-primary", "port": 3306, "user": "mmix", "password": "mmix", "database": "mmix"}, 100, 0.1)